# CERRA Projection

This notebooks create the CERRA area definition both from data downloaded
from the CDS API as well as from the projection invormations available at:

https://confluence.ecmwf.int/display/CKB/Copernicus+European+Regional+ReAnalysis+%28CERRA%29%3A+product+user+guide#CopernicusEuropeanRegionalReAnalysis(CERRA):productuserguide-TheCERRAgriddescription

Below are the essential parameters describing the grid and the used Lambert
Conformal Conic projection. More information about the grid and coordinates
can be found in the FAQ.

| Parameter                                 | Value       |
|-------------------------------------------|-------------|
| Number of points along x-axis             | 1069        |
| Number of points along y-axis             | 1069        |
| X-direction grid length                   | 5500 m      |
| Y-direction grid length                   | 5500 m      |
| Projection                                | Lambert Conformal Conic |
| Central meridian                          | 8           |
| Standard parallel 1                       | 50          |
| Standard parallel 2                       | 50          |
| Latitude of origin                        | 50          |
| Earth assumed spherical with radius       | 6371229 m   |

Latitude and longitude of the corner grid points in decimal degrees

| Grid point    | Latitude | Longitude |
|---------------|----------|-----------|
| Upper-left    | 63.7695  | -58.1051  |
| Upper-right   | 63.7695  | 74.1051   |
| Lower-right   | 20.2923  | 33.4859   |
| Lower-left    | 20.2923  | -17.4859  |

## From the documentation

In [ ]:
from pyresample.geometry import AreaDefinition
import pyproj

projparams = {
    'proj': 'lcc',
    'lat_0': 50,
    'lon_0': 8,
    'lat_1': 50,
    'lat_2': 50,
    'x_0': 0,
    'y_0': 0,
    'R': 6371229,
}

proj = pyproj.Proj(projparams)

# Convert corners back to geographical coordinates
lon_min, lat_min = -17.4859, 20.2929
lon_max, lat_max = 74.1051, 63.7695

x_min, y_min = proj(lon_min, lat_min, inverse=False)
x_max, y_max = proj(lon_max, lat_max, inverse=False)

# Print the area extent
area_extent_ll = lon_min, lat_min, lon_max, lat_max

area_extent = x_min, y_min, x_max, y_max
area_extent = tuple(round(value, 4) for value in area_extent)

print(
    "Geographical area extent (lon_min, lat_min, lon_max, lat_max): "
    f"{area_extent_ll}"
)

print(
    "Projection coordinates area extent (x_min, y_min, x_max, y_max): "
    f"{area_extent}"
)

area_id = 'cerra'
description = 'cerra'
proj_id = 'cerra'
projection = projparams
width = 1069
height = 1069
area_extent = area_extent
area_def = AreaDefinition(
    area_id,
    description,
    proj_id,
    projection,
    width,
    height,
    area_extent
)

print(area_def.dump())

## From grib data downloaded from the CDS API

In [ ]:
import pygrib
import pyproj

# Open the GRIB file
file_path = '/path/to/your/cerra/file/CERRA_ml_param_201802_an.grb'
fin = pygrib.open(file_path)

# Select the first GRIB message
grb = fin[1]

# Define Lambert Conformal projection
proj = pyproj.Proj(grb.projparams)

# Coordinates of the first grid point in projection space
x_min, y_min = proj(
    grb.longitudeOfFirstGridPointInDegrees,
    grb.latitudeOfFirstGridPointInDegrees
)

# Calculate the extent in projection coordinates
x_max, y_max = (
    x_min + (grb.Nx - 1) * grb.DxInMetres,
    y_min + (grb.Ny - 1) * grb.DyInMetres
)

# Convert corners back to geographical coordinates
lon_min, lat_min = proj(x_min, y_min, inverse=True)
lon_max, lat_max = proj(x_max, y_max, inverse=True)

# Print the area extent
area_extent_ll = (lon_min, lat_min, lon_max, lat_max)
area_extent_ll = tuple(round(value, 4) for value in area_extent_ll)

print(
    "Geographical area extent (lon_min, lat_min, lon_max, lat_max): "
    f"{area_extent_ll}"
)

area_extent = (x_min, y_min, x_max, y_max)
area_extent = tuple(round(value, 4) for value in area_extent)
print(
    "Projection coordinates area extent (x_min, y_min, x_max, y_max): "
    f"{area_extent}"
)

In [ ]:
from pyresample.geometry import AreaDefinition

area_id = 'cerra'
description = 'cerra'
proj_id = 'cerra'
projection = grb.projparams
width = grb.Nx
height = grb.Ny
area_extent = area_extent
area_def = AreaDefinition(
    area_id,
    description,
    proj_id,
    projection,
    width,
    height,
    area_extent
)

In [ ]:
print(area_def.dump())

## Testing

In [ ]:
import xarray as xr
import pyku

ds = xr.open_dataset('/path/to/your/cerra/file/CERRA_ml_param_201802_an.grb')
ds = ds.pyku.apply_georeferencing(area_def='cerra')
ds.pyku.get_area_def()

In [ ]:
ds.isel(time=0).pyku.select_area_extent(
     lower_left_lat=45.0,
     lower_left_lon=5.0,
     upper_right_lat=57.0,
     upper_right_lon=17.0,
).ana.one_map(var='q')